# Feature Engineering: Specs to Price
**Objective:** Transform raw technical specifications into meaningful market indicators.
**Input:** `cleaned_dataset.csv` (from Phase 03)

---

In [ ]:
import pandas as pd
import numpy as np
import os

# Input from Cleaning phase
input_path = '../03_Data_Preparation_Cleaning/cleaned_dataset.csv'
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} records for enrichment.")

In [ ]:
# 1. Screen Metrics
def extract_dimensions(res_str):
    try: return [int(x) for x in str(res_str).lower().split('x')]
    except: return [1920, 1080]

dims = df['SCREEN_RESOLUTION'].apply(extract_dimensions)
df['RES_WIDTH'] = [x[0] for x in dims]
df['RES_HEIGHT'] = [x[1] for x in dims]
df['TOTAL_PIXELS'] = df['RES_WIDTH'] * df['RES_HEIGHT']
df['PPI'] = np.sqrt(df['RES_WIDTH']**2 + df['RES_HEIGHT']**2) / df['SCREEN_SIZE']

# 2. Storage Score
df['STORAGE_SCORE'] = (df['SSD_GB'] * 1.0) + (df['HDD_GB'] * 0.2)

# 3. Tiers
def get_cpu_tier(cpu):
    cpu = str(cpu).upper()
    if any(x in cpu for x in ['I9', 'RYZEN 9', 'APPLE SILICON']): return 'Premium'
    if any(x in cpu for x in ['I7', 'RYZEN 7']): return 'High-End'
    if any(x in cpu for x in ['I5', 'RYZEN 5']): return 'Mid-Range'
    return 'Entry-Level'

df['CPU_TIER'] = df['CPU_BRAND'].apply(get_cpu_tier)

def get_brand_tier(brand):
    brand = str(brand).upper()
    if brand in ['APPLE', 'RAZER', 'MICROSOFT']: return 'Premium'
    if brand in ['HP', 'DELL', 'LENOVO', 'ASUS', 'MSI']: return 'Mainstream'
    return 'Budget'

df['BRAND_TIER'] = df['LAPTOP_BRAND'].apply(get_brand_tier)

# 4. Gaming Indicator
def identify_gaming(row):
    model = str(row['LAPTOP_MODEL']).upper()
    if any(x in model for x in ['ROG', 'TUF', 'LEGION', 'OMEN', 'PREDATOR', 'ALIENWARE']): return 1
    if 'NVIDIA RTX' in str(row['GPU_TYPE']): return 1
    return 0

df['IS_GAMING'] = df.apply(identify_gaming, axis=1)

print("Feature engineering completed.")

In [ ]:
# Export to Modeling folder
OUTPUT_FILE = '../06_Model_Training_Evaluation/final_cleaned_dataset.csv'
df.to_csv(OUTPUT_FILE, index=False)
df.to_csv('engineered_dataset.csv', index=False) # Keep a copy here
print(f"Enriched dataset saved to: {OUTPUT_FILE}")